In [1]:
import random

# Define a function that balances the dataset, k
def data_balancer(k, s=random.randint(1, 10)):
    
        # Separate the df into two subsets based on the labels column
        glitches = k[k['labels'] == 1.0]
        cleans = k[k['labels'] == 0.0]
        
        # Count the number of samples in each class
        num_glitches= len(glitches)
        num_cleans= len(cleans)
        # Print out the counts for user visibility
        print(f'Number of glitches:{num_glitches}')
        print(f'Number of cleans:{num_cleans}')
        # Determine the number of samples to keep from each class
        target_count = min(num_glitches, num_cleans)
        
        
        if num_glitches == target_count:
            # If glitches are already the minority class or both classes are equal
            df_cleans = cleans.sample(frac=target_count/num_cleans, random_state=s).reset_index(drop=True)
            df_glitches = glitches
        else:
            # Otherwise, downsample glitches
            df_glitches = glitches.sample(frac=target_count/num_glitches, random_state=s).reset_index(drop=True)
            df_cleans = cleans
            
        #glitches= glitches.sample(frac=1, random_state=s).reset_index(drop=True)
        #cleans=cleans.sample(frac=1, random_state=s).reset_index(drop=True)

        #if num_glitches == target_count:
            #df_cleans= cleans.iloc[:np.round(target_count).astype(int),:]
            #df_glitches=glitches
        #else:
            #df_glitches = glitches.iloc[:np.round(target_count).astype(int),:]
            #df_cleans=cleans

        df = pd.concat([df_glitches,df_cleans], ignore_index=True)
    
        return df

In [2]:
# List of data features
savedList=["fend", "tend", "q", "amplitude", "deltatime", "tstart", "fstart", "phase", "frequency", "snr"]
def filter_columns_by_ending(df, specific_ending):
    # Filter columns that end with the specific ending or are named 'labels'
    filtered_columns = [col for col in df.columns if (col.endswith(f'-{specific_ending}'))]
    #filtered_columns = [col for col in df.columns if (col.endswith(f'-{specific_ending}') or col.endswith('times') or col.endswith('labels'))]
    # Return a new DataFrame with only the filtered columns
    return df[filtered_columns]

In [3]:
# Replace all non-zero values with 1's
# Helpful for simplifying a dataset if you are only interested in the model learning presence, 
# ... not presence + magnitude of glitch values!
def replace_non_zero_with_one(df):
    # get columns that we need to keep in df
    excluded_columns = [col for col in df.columns if col.endswith(('times', 'labels'))]
    # copy the DataFrame to hold modified values, can't change df!
    df_modified = df.copy()
    # iterate through each column to replace non-zero values
    for column in df.columns:
        if column not in excluded_columns:
            # replace non-zero values with 1 at .where df[column] is 0
            df_modified[column] = df[column].where(df[column] == 0, 1)
    return df_modified

In [4]:
import tensorflow as tf
gpu_num = 0  # Desired GPU index

# Get a list of all the physical GPU devices available on the system
# This returns a list of tf.config.PhysicalDevice objects for devices of type 'GPU'
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        tf.config.set_visible_devices(gpus[gpu_num], 'GPU')
        print('Using GPU:', gpus[gpu_num])
    except RuntimeError as e:
        print(e)

2025-05-01 19:36:00.859240: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-01 19:36:00.874902: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-01 19:36:00.879717: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 19:36:00.891876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [5]:
import pandas as pd
# Import dataset here
# This was the dataset I used for reference -- LIGO O3 Observing Run data - All Glitch Types
df=pd.read_csv("/home/wadem/o3_omicron_triggers/all_glitch_data.csv")

In [6]:
# It takes forever to load in the dataset
# Save the df before testing so you dont overwrite it in pipleline
saved_df=df
print(df.shape)
print(saved_df.shape)

(271609, 8442)
(271609, 8442)


In [7]:
balanced_df = data_balancer(saved_df)

Number of glitches:13387
Number of cleans:258222


In [8]:
import numpy as np
# Check what percentage of the data frame are glitches
np.sum(balanced_df['labels'])/len(balanced_df)*100

50.0

In [9]:
# List of 3-letter subsystem codes
subsList = ["ASC", "CAL", "HPI", "IMC", "ISI", "LSC", "OMC", "PEM", "PSL", "SQZ", "SUS", "TCS"]

# Create and store the 12 subsystem datasets in a dictionary
subsystem_data = {}

# Loop through each subsystem and process columns related to it
for subsystem in subsList:
    # Filtering columns for the current subsystem
    filtered_columns = [
        col for col in balanced_df.columns
        # Include 'labels' column for all subsystems
        if col == "labels" or (":" in col and col.split(":")[1].startswith(subsystem))
    ]
    
    # Store the 2D DataFrame for this subsystem in the dictionary
    subsystem_data[subsystem] = balanced_df[filtered_columns]

# Now, subsystem_data will be a dictionary containing 2D DataFrames for each subsystem
df_ASC = subsystem_data["ASC"]
df_CAL = subsystem_data["CAL"]
df_HPI = subsystem_data["HPI"]
df_IMC = subsystem_data["IMC"]
df_ISI = subsystem_data["ISI"]
df_LSC = subsystem_data["LSC"]
df_OMC = subsystem_data["OMC"]
df_PEM = subsystem_data["PEM"]
df_PSL = subsystem_data["PSL"]
df_SQZ = subsystem_data["SQZ"]
df_SUS = subsystem_data["SUS"]
df_TCS = subsystem_data["TCS"]

In [10]:
# We can use this code to shorten the data set for efficient pipeline testing
# Not necessary to the model for official preditions, but great for debugging

factor = 40  # Keep every n row

# Shorten the dataset for test
# ::Factor is start/stop/step rows, : is all cols
df_ASC_short = df_ASC.iloc[::factor, :]

In [11]:
# Example usage:
# df is your original DataFrame
#specific_ending = 'fstart'  # Replace this with the ending you're looking for
#filtered_df = filter_columns_by_ending(df, specific_ending)
#times_data=filter_columns_by_ending(df, savedList[0])

#NOTE: Given new evidence supported in part by this project, it is likely that snr is the only 
# feature needed to get an accurate classification, so the other columns use a shortened dataset,
# but are ultimately irrelevant for use in the model - these are FEATURES not subsystems.
fend_data=filter_columns_by_ending(df_ASC_short, savedList[0])
tend_data=filter_columns_by_ending(df_ASC_short, savedList[1])
q_data=filter_columns_by_ending(df_ASC_short, savedList[2])
amp_data=filter_columns_by_ending(df_ASC_short, savedList[3])
dt_data=filter_columns_by_ending(df_ASC_short, savedList[4])
tstart_data=filter_columns_by_ending(df_ASC_short, savedList[5])
fstart_data=filter_columns_by_ending(df_ASC_short, savedList[6])
phase_data=filter_columns_by_ending(df_ASC_short, savedList[7])
freq_data=filter_columns_by_ending(df_ASC_short, savedList[8])
snr_data=filter_columns_by_ending(df_ASC, savedList[9])

In [12]:
# Dictionary to store SNR data for all subsystems
subsystems_SNR = {}

# Loop through each subsystem
for i in subsList:
    # Dynamically create the df name
    df_name = globals().get(f"df_{i}")
    
    # Filter the columns by "snr" for the specific subsystem df
    filtered_snr_data = filter_columns_by_ending(df_name, "snr")
    
    # Store the filtered df for each subsystem
    subsystems_SNR[i] = filtered_snr_data

In [ ]:
# DO NOT RUN UNLESS EXPORTING DATA
# Code to get subsystem CSVs for HPT

# Dictionary to store SNR data for all subsystems
# subsystems_SNR = {}

# Loop through each subsystem in the subsList
for i in subsList:
    # Dynamically create the DataFrame name (like df_ASC, df_CAL, etc.)
    df_name = globals().get(f"df_{i}")
    
    # Filter the columns by the suffix "snr" for the specific subsystem DataFrame
    filtered_snr_data = filter_columns_by_ending(df_name, "snr")
    
    # Store the filtered DataFrame for each subsystem
    subsystems_SNR[i] = filtered_snr_data

    # Assuming labels are stored in 'labels' column of balanced_df
    y = balanced_df['labels']
    
    # Set X as the filtered SNR data for this subsystem
    X = filtered_snr_data

    # Combine X and y into a single DataFrame
    df_combined = pd.DataFrame(X)
    df_combined['labels'] = y
    
    # If 'labels' is in X, drop it
    #if 'labels' in X.columns:
    #    X = X.drop(columns=['labels'])
    
    # print(df_combined.shape)
    
    # Save the combined DataFrame to a CSV file for this subsystem
    df_combined.to_csv(f'combined_data_{i}.csv', index=False)

    print(f"Processed and saved combined_data_{i}.csv")

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Subsystem abbreviations
subsList = ["ASC", "CAL", "HPI", "IMC", "ISI", "LSC", "OMC", "PEM", "PSL", "SQZ", "SUS", "TCS"]

# Hyperparameter dictionaries collected from runs - the runs for each subsystem are provided in the GitHub !
n_estimators_dict = {
    "ASC": 200, "CAL": 200, "HPI": 700, "IMC": 1000, "ISI": 700, "LSC": 1500,
    "OMC": 700, "PEM": 700, "PSL": 1500, "SQZ": 700, "SUS": 300, "TCS": 300
}

min_samples_split_dict = {
    "ASC": 10, "CAL": 20, "HPI": 5, "IMC": 2, "ISI": 5, "LSC": 10,
    "OMC": 5, "PEM": 5, "PSL": 10, "SQZ": 5, "SUS": 10, "TCS": 10
}

min_samples_leaf_dict = {
    "ASC": 6, "CAL": 6, "HPI": 6, "IMC": 6, "ISI": 6, "LSC": 8,
    "OMC": 8, "PEM": 8, "PSL": 8, "SQZ": 6, "SUS": 2, "TCS": 2
}

max_features_dict = {
    "ASC": "sqrt", "CAL": None, "HPI": "log2", "IMC": "log2", "ISI": "log2", "LSC": "sqrt",
    "OMC": "sqrt", "PEM": "sqrt", "PSL": "sqrt", "SQZ": "log2", "SUS": "log2", "TCS": "log2"
}

max_depth_dict = {
    "ASC": 20, "CAL": 10, "HPI": None, "IMC": 20, "ISI": None, "LSC": 10,
    "OMC": None, "PEM": None, "PSL": 10, "SQZ": None, "SUS": 60, "TCS": 60
}

bootstrap_dict = {
    "ASC": False, "CAL": True, "HPI": False, "IMC": True, "ISI": False, "LSC": True,
    "OMC": False, "PEM": False, "PSL": True, "SQZ": False, "SUS": True, "TCS": True
}

# Dictionary to store predictions for each subsystem
subsystem_preds = {}
subsystem_accuracies = {}

# Training loop
for subsystem in subsList:
    print(f"Training RF for subsystem: {subsystem}")
    
    # Extract data
    X = subsystems_SNR[subsystem]
    y = subsystem_data[subsystem]['labels']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    
    # Configure hyperparameters
    clf = RandomForestClassifier(
        n_estimators=n_estimators_dict[subsystem],
        random_state=42,
        max_features=max_features_dict[subsystem],
        max_depth=max_depth_dict[subsystem],
        min_samples_split=min_samples_split_dict[subsystem],
        min_samples_leaf=min_samples_leaf_dict[subsystem],
        bootstrap=bootstrap_dict[subsystem]
    )
    
    # Train
    clf.fit(X_train, y_train)
    
    # Predict
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    subsystem_accuracies[subsystem] = acc
    print(f"Accuracy for {subsystem}: {acc:.3f}")
    
    # Store predictions with correct index
    pred_df = pd.DataFrame({f"{subsystem}_pred": y_pred}, index=X_test.index)
    # Prediction df per subsysttem in this dictionary
    subsystem_preds[subsystem] = pred_df

# Combine all subsystem predictions into one DataFrame
# subsystem_preds.values() is list of all values in dictionary
# Combines them column-wise using axis=1
combined_preds_df = pd.concat(subsystem_preds.values(), axis=1)

# Preview combined dataset
print(combined_preds_df.head())

Training RF for subsystem: ASC
Accuracy for ASC: 0.795
Training RF for subsystem: CAL
Accuracy for CAL: 0.503
Training RF for subsystem: HPI
Accuracy for HPI: 0.590
Training RF for subsystem: IMC
Accuracy for IMC: 0.619
Training RF for subsystem: ISI
Accuracy for ISI: 0.584
Training RF for subsystem: LSC
Accuracy for LSC: 0.726
Training RF for subsystem: OMC
Accuracy for OMC: 0.514
Training RF for subsystem: PEM
Accuracy for PEM: 0.621
Training RF for subsystem: PSL
Accuracy for PSL: 0.535
Training RF for subsystem: SQZ
Accuracy for SQZ: 0.585
Training RF for subsystem: SUS
Accuracy for SUS: 0.560
Training RF for subsystem: TCS
Accuracy for TCS: 0.506
       ASC_pred  CAL_pred  HPI_pred  IMC_pred  ISI_pred  LSC_pred  OMC_pred  \
11755       1.0       1.0       1.0       0.0       1.0       1.0       0.0   
10248       1.0       1.0       1.0       1.0       1.0       0.0       0.0   
13408       0.0       1.0       0.0       0.0       0.0       0.0       0.0   
3087        1.0       1.

In [ ]:
# ONLY RUN FOR EXPORTING DATA

# Extract the label column (they are the same across subsystems)
any_subsystem = subsList[0]
labels_series = subsystem_data[any_subsystem]['labels'].loc[combined_preds_df.index]

# Add labels to the combined prediction DataFrame
combined_data_for_hpt = combined_preds_df.copy()
combined_data_for_hpt['labels'] = labels_series

# Export to CSV
combined_data_for_hpt.to_csv("combined_data_for_meta_rf.csv", index=False)

print("Exported combined data with labels to 'combined_data_for_meta_rf.csv'")

In [ ]:
# Best hyperparameters from tuning: {'n_estimators': 1500, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Grab all the subsystem prediction columns and turn them into a np array
# Each column is one subsystem, and each row is a signal instance
X_meta = combined_preds_df.values

# Pull out the true labels to go with our predictions
# Just grab them from any one of the subsystems since they're all the same
any_subsystem = list(subsystem_preds.keys())[0]
# We use the index from combined_preds_df to make sure labels line up with rows in X_meta
y_meta = subsystem_data[any_subsystem]['labels'].loc[combined_preds_df.index].values

# Split into training and test sets so we can evaluate performance properly
X_train, X_test, y_train, y_test = train_test_split(X_meta, y_meta, test_size=0.25, random_state=42)

# Set up the final Random Forest that will act as our "meta-classifier"
# This one learns how to combine the outputs from all the subsystem models

meta_clf = RandomForestClassifier(n_estimators=1500, # Number of trees
                             random_state=42, # Reproducibility
                             max_features="sqrt", # Limits feature selection per tree to prevent overfitting
                             max_depth=10, # Allow full tree growth initially
                             min_samples_split=10, # Minimum samples needed to split a node
                             min_samples_leaf=8, # Minimum samples required in a leaf node
                             bootstrap=True)

# Train the meta-classifier on the training data
meta_clf.fit(X_train, y_train)

# Get prediction probabilities on the test set (for ROC curve)
y_prob = meta_clf.predict_proba(X_test)[:, 1]

# Now we compute the false positive rate and true positive rate for the ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

# Calculate the Area Under the Curve (AUC) — a single number summarizing ROC performance
roc_auc = auc(fpr, tpr)

# Plot the ROC curve with labels and the AUC score
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"Meta RF ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')  # Diagonal line for reference
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Meta Random Forest')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# This would be how you curate the input X and Y data for running the 1 subsystem code
label_columns = [col for col in df_ASC.columns if 'labels' in col]
dtime_columns = [col for col in df_ASC.columns if 'deltatime' in col]
y = df_ASC[label_columns]
X = df_ASC.drop(columns=label_columns + dtime_columns)

In [ ]:
# OLD CODE
# 1 subsystem RF code
# Interesting to use for uncovering feature importance for a given subsystem

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train-test split
# Data gets shuffled, rs=42 to reproduce
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Initialize and train the Random Forest classifier
# From the documentation on tree depth: The maximum depth of the tree. 
# If None, then nodes are expanded until all leaves are pure or
# until all leaves contain less than min_samples_split samples.
# n_estimators = num of trees

# Params from 200 iteration HPT on elrond

clf = RandomForestClassifier(n_estimators=1000, # Number of trees
                             random_state=42, # Reproducibility
                             max_features="log2", # Limits feature selection per tree to prevent overfitting
                             max_depth=30, # Allow full tree growth initially
                             min_samples_split=5, # Minimum samples needed to split a node
                             min_samples_leaf=1, # Minimum samples required in a leaf node
                             bootstrap=True)

# Build a forest of trees from the training set (X, y)
clf.fit(X_train, y_train)

# Predictions
# Predict class for X
y_pred = clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.3f}")

# Show feature importances
# The importance score is a relative value—higher values indicate greater influence on the model
# clf.feature_importances_: This is an array where each value 
# represents the importance score of a corresponding feature.
feature_importances = pd.DataFrame({'Feature': X.columns, 'Importance': clf.feature_importances_})
# Sort the feature_importances with the most important feature first
print(feature_importances.sort_values(by='Importance', ascending=False))

In [ ]:
# 1 subsystem ROC Original Code
# Keeping this in my code for reference to how I originally plotted the 1 subsystem ROC during development

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# roc_curve:

# Returns false positive rate fpr: Increasing false positive rates such that element i 
# is the false positive rate of predictions with score >= thresholds[i].

# Returns true positive rate fpr: Increasing true positive rates such that element i 
# is the true positive rate of predictions with score >= thresholds[i].

# Returns thresholds from the predicted probabilities


# Predict class probabilities for X_test
# Returns array of (n_samples, n_classes)
# Grabs all rows and second column to know 1D array of probs
y_pred_prob_rf = clf.predict_proba(X_test)[:, 1]

# Compute ROC curve
# 
fpr_rf, tpr_rf, thresholds = roc_curve(y_test, y_pred_prob_rf)

# Compute AUC from auc(x, y)
roc_auc_rf = auc(fpr_rf, tpr_rf)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, color='blue', lw=2, label=f'Random Forest (AUC = {roc_auc_rf:.3f})')
plt.axline((0, 0), (1, 1), color='gray', linestyle='--', lw=2)  # No skill line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()